In [ ]:
# Cell 0: Patch numpy 2.x → 1.x compatibility + transformers
import os, glob, sys, importlib
from pathlib import Path

# === NumPy compatibility shim ===
import numpy as np
import numpy._core._multiarray_umath as _mm

if not hasattr(_mm, '_blas_supports_fpe'):
    def _blas_fpe(x): return False
    _mm._blas_supports_fpe = _blas_fpe
    print('[patch] Added numpy._blas_supports_fpe', flush=True)

import numpy._core.umath as _um
if not hasattr(_um, '_center'):
    try:
        from numpy._core.strings import _center as _cfunc
        _um._center = _cfunc
        print('[patch] Added numpy._center', flush=True)
    except ImportError:
        _um._center = lambda a, w, fc=' ': a
        print('[patch] Added numpy._center stub', flush=True)

import numpy.strings as _str
_str_funcs = {'partition':'partition','split':'split','find':'find','rfind':'rfind','count':'count','startswith':'startswith','endswith':'endswith'}
for _sname, _dname in _str_funcs.items():
    if not hasattr(_str, _sname):
        try:
            _dmod = importlib.import_module('numpy._core.defchararray')
            if hasattr(_dmod, _dname): setattr(_str, _sname, getattr(_dmod, _dname))
            else: setattr(_str, _sname, lambda *a, **k: None)
        except ImportError:
            setattr(_str, _sname, lambda *a, **k: None)
        print(f'[patch] Added numpy.strings.{_sname}', flush=True)

# === Transformer patches ===
for pg in glob.glob('/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py'):
    s = Path(pg).read_text()
    if 'split_attention_implementation' not in s:
        idx = s.find(chr(10)+'def ')
        if idx < 0: idx = s.rfind(chr(10)+'class '); idx = s.find(chr(10), idx) if idx >= 0 else -1
        if idx < 0: idx = len(s)
        s = s[:idx]+chr(10)+'def split_attention_implementation() -> str:'+chr(10)+'    return'+chr(39)+'flash_attention_2'+chr(39)+chr(10)+s[idx:]
        Path(pg).write_text(s)
        print('[patch] Patched generic.py', flush=True)

for pu in glob.glob('/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py'):
    s = Path(pu).read_text()
    if 'KERNELS_MAX_VERSION' not in s:
        idx = s.find(chr(10)+'def ')
        if idx < 0: idx = s.rfind(chr(10)+'class '); idx = s.find(chr(10), idx) if idx >= 0 else -1
        if idx < 0: idx = len(s)
        s = s[:idx]+chr(10)+'def KERNELS_MAX_VERSION() -> str:'+chr(10)+'    return'+chr(39)+'0.4.0'+chr(39)+chr(10)+s[idx:]
        Path(pu).write_text(s)
        print('[patch] Patched import_utils.py: KERNELS_MAX_VERSION', flush=True)

for pa in ['/usr/local/lib/python3.12/dist-packages/transformers/utils/auto_docstring.py']:
    if os.path.exists(pa):
        s = Path(pa).read_text()
        target = 'def auto_class_docstring(cls, custom_intro=None, custom_args=None, checkpoint=None):'
        if target in s and 'except ValueError' not in s:
            def_pos = s.find(target); rest = s[def_pos+len(target):]; lines = rest.split(chr(10))
            first_code = None
            for i,line in enumerate(lines):
                st = line.strip()
                if st and not st.startswith('#'): first_code = line; break
            if first_code:
                indent = ' '*(len(first_code)-len(first_code.lstrip()))
                nb = indent+'try:'+chr(10)+indent+'    '+first_code.strip()+chr(10)+indent+'except ValueError:'+chr(10)+indent+'    return cls'+chr(10)+chr(10)
                s = s[:def_pos+len(target)]+chr(10)+nb+s[def_pos+len(target)+len(first_code):]
                Path(pa).write_text(s)
                print('[patch] Patched auto_docstring.py', flush=True)

for d in ['/usr/local/lib/python3.12/dist-packages/transformers','/usr/local/lib/python3.12']:
    for f in glob.glob(d+'/**/*.pyc', recursive=True):
        try: os.remove(f)
        except: pass
print('[patch] Cleared .pyc cache', flush=True)

for mn, attrs in [('transformers.utils.generic', {'split_attention_implementation': lambda:'flash_attention_2'}), ('transformers.utils.import_utils', {'KERNELS_MAX_VERSION': lambda:'0.4.0'}), ('transformers.utils', {'KERNELS_MAX_VERSION': lambda:'0.4.0'})]:
    m = sys.modules.get(mn)
    if m:
        for a,v in attrs.items():
            if not hasattr(m,a): setattr(m,a,v)
        print(f'[patch] Injected {mn}', flush=True)

if 'transformers' in sys.modules:
    tu = sys.modules['transformers'].utils; orig = tu.auto_docstring
    def patched(*a,**kw):
        d = orig(*a,**kw)
        def w(cls):
            try: return d(cls)
            except ValueError: return cls
        return w
    tu.auto_docstring = patched
    print('[patch] Wrapped auto_docstring', flush=True)

print('[patch] All patches complete', flush=True)


In [ ]:
!pip install -q git+https://github.com/jaredpalmer/kev.git 2>&1 | tail -3
print('pip install kev done', flush=True)

import os, torch
print(f'torch: {torch.__version__}', flush=True)
print(f'cuda: {torch.cuda.is_available()}', flush=True)
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}', flush=True)

import kev, kev.model
print(f'kev imported: {kev.__file__}', flush=True)

# Check available Qwen models in transformers
from transformers import AutoModelForCausalLM
qwen_models = [m for m in dir(__import__('transformers').models) if 'qwen' in m.lower()]
print(f'Available Qwen modules: {qwen_models}', flush=True)

# Try Qwen2 (not Qwen2.5 - not in this transformers version)
for mn in ['Qwen/Qwen2-0.5B','Qwen/Qwen2-7B']:
    try:
        m = kev.model.AutoModelForCausalLM.from_pretrained(mn, trust_remote_code=True)
        print(f'Model loaded: {mn} ({type(m).__name__})', flush=True)
        del m; torch.cuda.empty_cache(); break
    except Exception as e:
        print(f'Failed {mn}: {e}', flush=True)
        torch.cuda.empty_cache()

print('Cell 2 done', flush=True)


In [ ]:
import os, glob
from pathlib import Path

od = '/tmp/kev-wcag-model'
os.makedirs(od, exist_ok=True)
print(f'output_dir: {od}', flush=True)
if os.path.isdir(od):
    for f in sorted(Path(od).glob('**/*')):
        if f.is_file():
            print(f'  {f.relative_to(od)}: {os.path.getsize(f)/1024:.1f} KB', flush=True)

print('Cell 3 done', flush=True)
